# MDD + Antidepressant Patient Characteristics

This notebook summarizes demographic characteristics and follow-up timing for patients with MDD and antidepressant exposure.

Main outputs include patient count, age distribution, gender, race, ethnicity, follow-up days after first MDD diagnosis, and index year distribution.


## Import person-level dataset

This All of Us Workbench query imports person-level demographic variables for the MDD + antidepressant cohort.


In [ ]:
import pandas
import os

# This query represents dataset "Demo_Patients with MDD + Antidepressants (N=68301)" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_60565800_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (4152280)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT ca.descendant_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                    JOIN
                        (SELECT
                            DISTINCT c.concept_id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id             
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                            WHERE
                                concept_id IN (798834, 35603277, 715939, 766209, 713109, 46275300, 721724, 781705, 755695, 754270, 738156, 715259, 739138, 1510996, 722031, 37498659, 797617, 778268, 43560354, 743670, 750982, 733896, 751412, 710062, 703547, 705755, 725131, 40234834, 703470, 716968, 44507700, 1366610, 714684, 717607)             
                                AND full_text LIKE '%_rank1]%'       ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) b 
                            ON (ca.ancestor_id = b.concept_id)) 
                        AND is_standard = 1)) criteria ) )"""

dataset_60565800_person_df = pandas.read_gbq(
    dataset_60565800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_60565800_person_df.head(20)

## Initial cohort count

This step checks the total number of unique patients in the person-level dataset.


In [ ]:
import pandas as pd
import numpy as np

demo_df = dataset_60565800_person_df.copy()

# Check total unique patients
demo_n = demo_df["person_id"].nunique()
print("N =", demo_n)

## Age calculation

Age is calculated using a fixed reference date so the table can be reproduced consistently.


In [ ]:
reference_date = pd.to_datetime("2023-10-01")

demo_df["date_of_birth"] = pd.to_datetime(demo_df["date_of_birth"], errors="coerce", utc=True)
demo_df["birth_date"] = demo_df["date_of_birth"].dt.date

demo_df["age_int"] = (
    reference_date.year 
    - pd.to_datetime(demo_df["birth_date"]).dt.year
    - (
        (reference_date.month < pd.to_datetime(demo_df["birth_date"]).dt.month) |
        (
            (reference_date.month == pd.to_datetime(demo_df["birth_date"]).dt.month) &
            (reference_date.day < pd.to_datetime(demo_df["birth_date"]).dt.day)
        )
    ).astype(int)
)

age_mean = round(demo_df["age_int"].mean(), 1)
age_median = round(demo_df["age_int"].median(), 1)
age_min = int(demo_df["age_int"].min())
age_max = int(demo_df["age_int"].max())

age_rows = pd.DataFrame({
    "Category": ["mean", "median", "min, max"],
    "Count": [age_mean, age_median, f"{age_min}, {age_max}"]
})

age_rows

## Demographic variable cleaning

The following helper functions standardize ethnicity, gender, and race categories for reporting.


In [ ]:


def clean_ethnicity(x):
    if pd.isna(x):
        return "Missing"
    
    x_lower = str(x).lower()
    
    if "hispanic or latino" in x_lower and "not hispanic" not in x_lower:
        return "Hispanic or Latino"
    
    if "not hispanic or latino" in x_lower:
        return "Not Hispanic or Latino"
    
    if "no matching concept" in x_lower:
        return "No matching concept"
    
    if "prefer not" in x_lower:
        return "Prefer Not To Answer"
    
    if "race ethnicity none of these" in x_lower or "none of these" in x_lower:
        return "Race Ethnicity None Of These"
    
    if "skip" in x_lower:
        return "Skip"
    
    return "Other"


ethnicity_order = [
    "Hispanic or Latino",
    "No matching concept",
    "Not Hispanic or Latino",
    "Prefer Not To Answer",
    "Race Ethnicity None Of These",
    "Skip"
]

demo_df["ethnicity_clean"] = demo_df["ethnicity"].apply(clean_ethnicity)

ethnicity_counts = (
    demo_df
    .groupby("ethnicity_clean")["person_id"]
    .nunique()
)

ethnicity_rows = pd.DataFrame({
    "Category": ethnicity_order,
    "Count": [int(ethnicity_counts.get(x, 0)) for x in ethnicity_order]
})

ethnicity_rows

In [ ]:
def clean_gender(x):
    if pd.isna(x):
        return "Unknown"
    
    x_lower = str(x).lower().strip()
    
    if x_lower == "female":
        return "Female"
    
    if x_lower == "male":
        return "Male"
    
    if "unknown" in x_lower or "no matching concept" in x_lower:
        return "Unknown"
    
    # Everything else goes into this combined category
    return "Not man only, not woman only, prefer not to answer, or skipped"


gender_order = [
    "Female",
    "Male",
    "Not man only, not woman only, prefer not to answer, or skipped",
    "Unknown"
]

demo_df["gender_clean"] = demo_df["gender"].apply(clean_gender)

gender_counts = (
    demo_df
    .groupby("gender_clean")["person_id"]
    .nunique()
)

gender_rows = pd.DataFrame({
    "Category": gender_order,
    "Count": [int(gender_counts.get(x, 0)) for x in gender_order]
})

gender_rows

In [ ]:
def clean_race_full(x):
    if pd.isna(x):
        return "None Indicated"
    
    x_lower = str(x).lower().strip()
    
    if "american indian or alaska native" in x_lower:
        return "American Indian or Alaska Native"
    
    if "asian" in x_lower:
        return "Asian"
    
    if "black or african american" in x_lower:
        return "Black or African American"
    
    if "prefer not" in x_lower:
        return "I prefer not to answer"
    
    if "middle eastern or north african" in x_lower:
        return "Middle Eastern or North African"
    
    if "more than one population" in x_lower:
        return "More than one population"
    
    if "native hawaiian or other pacific islander" in x_lower:
        return "Native Hawaiian or Other Pacific Islander"
    
    if "none indicated" in x_lower:
        return "None Indicated"
    
    if "none of these" in x_lower:
        return "None of these"
    
    if "skip" in x_lower:
        return "Skip"
    
    if "white" in x_lower:
        return "White"
    
    return "Other"


race_order_full = [
    "American Indian or Alaska Native",
    "Asian",
    "Black or African American",
    "I prefer not to answer",
    "Middle Eastern or North African",
    "More than one population",
    "Native Hawaiian or Other Pacific Islander",
    "None Indicated",
    "None of these",
    "Skip",
    "White"
]

demo_df["race_clean_full"] = demo_df["race"].apply(clean_race_full)

race_counts_full = (
    demo_df
    .groupby("race_clean_full")["person_id"]
    .nunique()
)

race_rows_full = pd.DataFrame({
    "Category": race_order_full,
    "Count": [int(race_counts_full.get(x, 0)) for x in race_order_full]
})

race_rows_full
    

In [ ]:
demo_df["race"].value_counts(dropna=False)

In [ ]:
import pandas
import os

# This query represents dataset "Demo_Condi_MDD + antidepressants (N= 68301)" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_05646614_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4152280)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (4152280)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (798834, 35603277, 715939, 766209, 713109, 46275300, 721724, 781705, 755695, 754270, 738156, 715259, 739138, 1510996, 722031, 37498659, 797617, 778268, 43560354, 743670, 750982, 733896, 751412, 710062, 703547, 705755, 725131, 40234834, 703470, 716968, 44507700, 1366610, 714684, 717607)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_05646614_condition_df = pandas.read_gbq(
    dataset_05646614_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_05646614_condition_df.head(5)

## Import MDD condition records

This query imports MDD condition occurrence records used to identify each patient’s first observed MDD diagnosis date.


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Use your tbl2 demo datasets
# -----------------------------
demo_person_df = dataset_60565800_person_df.copy()
mdd_condition_df = dataset_05646614_condition_df.copy()

# Base cohort N
BASE_N = 68301

# Controlled Tier v8 cutoff date
reference_date = pd.to_datetime("2023-10-01").date()

# -----------------------------
# Make sure condition table only includes people in demo cohort
# -----------------------------
demo_person_ids = demo_person_df["person_id"].dropna().unique()

mdd_condition_df = mdd_condition_df[
    mdd_condition_df["person_id"].isin(demo_person_ids)
].copy()

# -----------------------------
# Convert condition_start_datetime to date only
# -----------------------------
mdd_condition_df["condition_start_datetime"] = pd.to_datetime(
    mdd_condition_df["condition_start_datetime"],
    errors="coerce",
    utc=True
)

mdd_condition_df["condition_start_date"] = (
    mdd_condition_df["condition_start_datetime"].dt.date
)

# -----------------------------
# Get first MDD diagnosis date for each person
# -----------------------------
first_mdd_df = (
    mdd_condition_df
    .dropna(subset=["condition_start_date"])
    .groupby("person_id", as_index=False)["condition_start_date"]
    .min()
    .rename(columns={"condition_start_date": "first_mdd_date"})
)

print("Expected cohort N:", BASE_N)
print("Patients with first MDD date:", first_mdd_df["person_id"].nunique())

## Build demographic and index-date summary

This section combines person-level information with first MDD diagnosis date and prepares descriptive summary tables.


In [ ]:
# -----------------------------
# FU days after first MDD date
# Definition: 2023-10-01 - first MDD diagnosis date
# -----------------------------
first_mdd_df["fu_days_after_first_mdd"] = (
    pd.to_datetime(reference_date) - pd.to_datetime(first_mdd_df["first_mdd_date"])
).dt.days

fu_mean = round(first_mdd_df["fu_days_after_first_mdd"].mean(), 1)
fu_median = round(first_mdd_df["fu_days_after_first_mdd"].median(), 1)
fu_min = int(first_mdd_df["fu_days_after_first_mdd"].min())
fu_max = int(first_mdd_df["fu_days_after_first_mdd"].max())

fu_days_rows = pd.DataFrame({
    "Category": ["mean", "median", "min, max"],
    "Count": [fu_mean, fu_median, f"{fu_min}, {fu_max}"]
})

fu_days_rows

In [ ]:
# -----------------------------
# Index year at first MDD Dx date
# -----------------------------
first_mdd_df["index_year"] = pd.to_datetime(
    first_mdd_df["first_mdd_date"]
).dt.year

year_counts = (
    first_mdd_df
    .groupby("index_year")["person_id"]
    .nunique()
)

# Use 1980 to 2023 to match your Excel template
year_rows = pd.DataFrame({
    "Category": list(range(1980, 2024)),
    "Count": [int(year_counts.get(year, 0)) for year in range(1980, 2024)]
})

index_year_rows = pd.concat(
    [
        pd.DataFrame({
            "Category": ["n of PT"],
            "Count": [first_mdd_df["person_id"].nunique()]
        }),
        year_rows
    ],
    ignore_index=True
)

index_year_rows